# AIOps Lab — Supervised Learning: Alert Priority Classifier
### Using Elastic Data Frame Analytics (Classification)
**AIOps Fundamentals Training | Microland**

---

## What This Lab Demonstrates

This lab demonstrates **supervised machine learning** using Elastic's
**Data Frame Analytics — Classification** feature.

Unlike the anomaly detection labs (which use unsupervised ML and need no labels),
this lab uses **labelled historical data** — alerts where we already know
the outcome — to train a model that can predict the priority of NEW alerts
automatically.

---

## The Business Problem

Your L1 operations team receives hundreds of alerts per day.
Every alert arrives with the same default priority: **Unknown**.
An L1 engineer must read each alert, assess the context, and decide:
is this a P1 (Critical), P2 (High), or P3 (Medium) incident?

This triage process takes time — and when a genuine P1 is misread as a P3,
the consequences are severe.

**The AIOps solution:** Train a classification model on 500 historical alerts
where we know the correct priority. The model learns which combinations of
features (CPU%, memory%, error rate, affected services, time of day, server type)
typically produce P1 vs P2 vs P3 incidents. Then it automatically predicts
the priority of every new incoming alert — before a human reads it.

---

## Supervised vs Unsupervised — The Key Distinction

| Approach | Needs Labels? | AIOps Use Case | This Lab? |
|----------|--------------|----------------|----------|
| **Unsupervised ML** (anomaly detection) | No | Detect unusual patterns in live data | Anomaly detection labs (Days 2-4) |
| **Supervised ML** (classification) | **Yes** — labelled examples required | Predict category of new data based on past examples | **This lab** |
| **Supervised ML** (regression) | **Yes** — labelled examples required | Predict a numeric value (e.g. MTTR in minutes) | Extension section |

---

## What the Model Will Learn

Training data features (inputs):
- `cpu_pct` — CPU utilisation at the time of the alert
- `memory_pct` — Memory utilisation at the time of the alert
- `disk_pct` — Disk utilisation at the time of the alert
- `error_rate` — Application error rate (errors per minute)
- `affected_services` — Number of downstream services impacted
- `response_time_ms` — Application response time in milliseconds
- `hour_of_day` — Hour when the alert fired (0-23)
- `is_business_hours` — Whether the alert occurred during business hours
- `server_type` — Type of server (app_server, db_server, web_server, etc.)
- `alert_source` — What triggered the alert (cpu_monitor, memory_monitor, etc.)
- `previous_incidents_7d` — Number of incidents on this server in the last 7 days

Label (what we are predicting):
- `incident_priority` — P1 (Critical), P2 (High), or P3 (Medium)

---

## Before You Start
1. Elastic Cloud deployment running
2. Cloud ID and API Key ready
3. Running in Google Colab or Jupyter

> **No Python knowledge required.** Run each cell in order.


## Step 1 — Install Libraries


In [25]:
!pip install elasticsearch pandas --quiet
print('All Libraries are Installed!!....Libraries ready.')


All Libraries are Installed!!....Libraries ready.


## Step 2 — Configuration


In [ ]:
# -----------------------------------------------------------
# ELASTIC CLOUD CREDENTIALS
# -----------------------------------------------------------
CLOUD_ID   = 'Your Cloud ID'
API_KEY    = 'Your API key'

# Index names
TRAINING_INDEX   = 'aiops-alert-training'    # 500 labelled historical alerts
INFERENCE_INDEX  = 'aiops-alert-new'         # 50 new unlabelled alerts to predict
DFA_JOB_ID       = 'alert-priority-classifier'
MODEL_ID         = 'alert-priority-classifier-1'   # Elastic auto-names the model

print('Configuration set.')
print(f'   Training index  : {TRAINING_INDEX}  (500 labelled alerts)')
print(f'   Inference index : {INFERENCE_INDEX} (50 new alerts to classify)')
print(f'   DFA Job ID      : {DFA_JOB_ID}')


## Step 3 — Connect to Elastic Cloud


In [ ]:
from elasticsearch import Elasticsearch

es = Elasticsearch(cloud_id=CLOUD_ID, api_key=API_KEY)
info = es.info()
print(f'Connected: {info["cluster_name"]} v{info["version"]["number"]}')


## Step 4 — Create the Training and Inference Indices

We need two indices:
1. **Training index** — 500 historical alerts WITH the correct priority label
2. **Inference index** — 50 new alerts WITHOUT a label (model will predict these)

The field `incident_priority` is our **label field** — the answer the model learns to predict.
In the inference index this field is left empty — the model fills it in.


In [ ]:
for index_name in [TRAINING_INDEX, INFERENCE_INDEX]:
    if es.indices.exists(index=index_name):
        es.indices.delete(index=index_name)
        print(f'Deleted existing: {index_name}')

mapping = {
    'mappings': {
        'properties': {
            'alert_id':              {'type': 'keyword'},
            '@timestamp':            {'type': 'date'},
            'host_name':             {'type': 'keyword'},
            'server_type':           {'type': 'keyword'},
            'alert_source':          {'type': 'keyword'},
            'cpu_pct':               {'type': 'float'},
            'memory_pct':            {'type': 'float'},
            'disk_pct':              {'type': 'float'},
            'error_rate':            {'type': 'float'},
            'affected_services':     {'type': 'integer'},
            'response_time_ms':      {'type': 'float'},
            'hour_of_day':           {'type': 'integer'},
            'is_business_hours':     {'type': 'boolean'},
            'day_of_week':           {'type': 'keyword'},
            'previous_incidents_7d': {'type': 'integer'},
            'incident_priority':     {'type': 'keyword'},   # LABEL FIELD
            'actual_downtime_min':   {'type': 'float'},     # for regression extension
            'is_training_data':      {'type': 'boolean'}
        }
    }
}

es.indices.create(index=TRAINING_INDEX,  body=mapping)
es.indices.create(index=INFERENCE_INDEX, body=mapping)
print(f'Created: {TRAINING_INDEX}')
print(f'Created: {INFERENCE_INDEX}')
print('   Field incident_priority is the LABEL the model will learn to predict.')


## Step 5 — Generate the Labelled Training Dataset (500 Alerts)

This is the most important cell in the lab. We create 500 realistic historical alerts
where we already **know the correct priority**. This is the labelled data that makes
supervised learning possible.

### How the Labels Were Decided (Business Rules)

In a real organisation, these labels come from historical ITSM records — past incidents
where engineers already classified the priority. For this lab, we simulate that by
applying realistic business rules:

| Priority | Criteria | Real-World Meaning |
|----------|----------|--------------------|
| **P1 Critical** | CPU > 90% OR memory > 92% OR 3+ services affected OR error rate > 50/min | Service down or imminent crash — all hands |
| **P2 High** | CPU 75-90% OR memory 80-92% OR 2 services affected OR error rate 20-50/min | Degraded service — escalate within 30 min |
| **P3 Medium** | Everything else | Warning — monitor and investigate |

### The Realistic Noise

Real incident data is never perfectly clean. We add:
- **Label noise (8%):** Sometimes a P2 situation gets labelled P1 because it happened
  at 3 AM and the on-call engineer was cautious — or a P1 was logged as P2 because
  the engineer was busy. The model must learn to handle this.
- **Feature noise:** Metrics have natural variation — they are not perfectly consistent
  for the same priority level every time.

This is what makes the training data realistic rather than toy data.


In [ ]:
import random
from datetime import datetime, timedelta, timezone
import pandas as pd

random.seed(42)

SERVER_TYPES   = ['app_server', 'db_server', 'web_server', 'file_server', 'dc_server']
ALERT_SOURCES  = ['cpu_monitor', 'memory_monitor', 'disk_monitor',
                  'app_error_monitor', 'network_monitor', 'service_monitor']
DAYS_OF_WEEK   = ['Monday', 'Tuesday', 'Wednesday', 'Thursday',
                  'Friday', 'Saturday', 'Sunday']
HOSTS          = [f'SRV-{i:02d}' for i in range(1, 21)]  # 20 servers

now = datetime.now(timezone.utc)


def generate_alert(alert_id, is_training=True):
    """Generate one realistic IT alert with consistent feature-label relationships."""

    server_type   = random.choice(SERVER_TYPES)
    alert_source  = random.choice(ALERT_SOURCES)
    hour_of_day   = random.randint(0, 23)
    day_idx       = random.randint(0, 6)
    day_of_week   = DAYS_OF_WEEK[day_idx]
    is_biz_hours  = (8 <= hour_of_day <= 18) and (day_idx < 5)
    prev_incidents = random.randint(0, 8)
    ts = now - timedelta(
        days=random.randint(1, 90),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59)
    )

    # ── Decide priority first, then generate realistic metrics for it ──
    # This ensures the training data has clear feature-label relationships
    # that the model can learn from

    # Weighted random: 20% P1, 35% P2, 45% P3 (realistic distribution)
    rand_priority = random.random()
    if rand_priority < 0.20:
        priority = 'P1'
    elif rand_priority < 0.55:
        priority = 'P2'
    else:
        priority = 'P3'

    # Generate metrics consistent with the priority
    if priority == 'P1':
        cpu_pct            = random.uniform(88, 99)
        memory_pct         = random.uniform(85, 99)
        disk_pct           = random.uniform(70, 95)
        error_rate         = random.uniform(45, 200)
        affected_services  = random.randint(2, 8)
        response_time_ms   = random.uniform(3000, 15000)
        actual_downtime    = random.uniform(30, 480)

    elif priority == 'P2':
        cpu_pct            = random.uniform(70, 90)
        memory_pct         = random.uniform(72, 88)
        disk_pct           = random.uniform(60, 82)
        error_rate         = random.uniform(15, 55)
        affected_services  = random.randint(1, 3)
        response_time_ms   = random.uniform(1000, 4000)
        actual_downtime    = random.uniform(10, 60)

    else:  # P3
        cpu_pct            = random.uniform(40, 75)
        memory_pct         = random.uniform(45, 75)
        disk_pct           = random.uniform(40, 65)
        error_rate         = random.uniform(0, 20)
        affected_services  = random.randint(0, 2)
        response_time_ms   = random.uniform(200, 1200)
        actual_downtime    = random.uniform(0, 15)

    # Add 8% label noise (realistic human classification errors)
    if is_training and random.random() < 0.08:
        priorities = ['P1', 'P2', 'P3']
        priorities.remove(priority)
        priority = random.choice(priorities)

    return {
        'alert_id':              f'ALT-{alert_id:04d}',
        '@timestamp':            ts.isoformat(),
        'host_name':             random.choice(HOSTS),
        'server_type':           server_type,
        'alert_source':          alert_source,
        'cpu_pct':               round(cpu_pct, 2),
        'memory_pct':            round(memory_pct, 2),
        'disk_pct':              round(disk_pct, 2),
        'error_rate':            round(error_rate, 2),
        'affected_services':     affected_services,
        'response_time_ms':      round(response_time_ms, 1),
        'hour_of_day':           hour_of_day,
        'is_business_hours':     is_biz_hours,
        'day_of_week':           day_of_week,
        'previous_incidents_7d': prev_incidents,
        'incident_priority':     priority if is_training else None,
        'actual_downtime_min':   round(actual_downtime, 1),
        'is_training_data':      is_training
    }


# Generate 500 labelled training alerts
training_alerts = [generate_alert(i, is_training=True) for i in range(1, 501)]

# Generate 50 unlabelled inference alerts (no priority label)
inference_alerts = [generate_alert(i + 500, is_training=False) for i in range(1, 51)]

# Summary
df_train = pd.DataFrame(training_alerts)
p1 = (df_train['incident_priority'] == 'P1').sum()
p2 = (df_train['incident_priority'] == 'P2').sum()
p3 = (df_train['incident_priority'] == 'P3').sum()

print('Training dataset generated:')
print(f'   Total alerts     : {len(training_alerts)}')
print(f'   P1 (Critical)    : {p1}  ({p1/len(training_alerts)*100:.0f}%)')
print(f'   P2 (High)        : {p2}  ({p2/len(training_alerts)*100:.0f}%)')
print(f'   P3 (Medium)      : {p3}  ({p3/len(training_alerts)*100:.0f}%)')
print()
print(f'Inference dataset  : {len(inference_alerts)} alerts (no priority label)')
print('The model will predict the priority for each of these 50 alerts.')


## Step 6 — Explore the Training Data

Before uploading, explore the data to understand what the model will learn.
Notice how the feature values differ clearly between P1, P2, and P3 alerts.
These patterns are what the classification algorithm will learn.


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

df = pd.DataFrame(training_alerts)
colors_map = {'P1': '#C55A11', 'P2': '#0D6E6E', 'P3': '#1B3A6B'}

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

features = [
    ('cpu_pct',           'CPU Utilisation %'),
    ('memory_pct',        'Memory Utilisation %'),
    ('error_rate',        'Error Rate (errors/min)'),
    ('affected_services', 'Affected Services Count'),
    ('response_time_ms',  'Response Time (ms)'),
    ('previous_incidents_7d', 'Previous Incidents (7 days)'),
]

for idx, (feature, label) in enumerate(features):
    ax = axes[idx]
    for priority in ['P3', 'P2', 'P1']:
        subset = df[df['incident_priority'] == priority][feature]
        ax.hist(subset, bins=20, alpha=0.65,
                color=colors_map[priority], label=priority)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel('Alert Count', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle(
    'Feature Distribution by Incident Priority\n'
    'Clear separation = the model will learn this pattern reliably',
    fontsize=12, fontweight='bold', color='#1B3A6B'
)
plt.tight_layout()
plt.show()

# Print mean values per priority
print('Mean feature values by priority:')
print('-' * 70)
summary = df.groupby('incident_priority')[[
    'cpu_pct', 'memory_pct', 'error_rate',
    'affected_services', 'response_time_ms'
]].mean().round(1)
print(summary.to_string())
print()
print('Notice: P1 alerts have clearly higher values across almost every feature.')
print('This clear separation is what makes classification work reliably.')


## Step 7 — Upload Training and Inference Data to Elastic


In [ ]:
from elasticsearch.helpers import bulk

def generate_actions(docs, index):
    for doc in docs:
        # Remove None values — Elastic DFA needs clean numeric fields
        clean = {k: v for k, v in doc.items() if v is not None}
        yield {'_index': index, '_source': clean}

# Upload training data
print(f'Uploading {len(training_alerts)} labelled training alerts...')
success, errors = bulk(es, generate_actions(training_alerts, TRAINING_INDEX),
                       chunk_size=100, raise_on_error=False)
print(f'   Training index: {success} documents indexed, {len(errors)} errors')

# Upload inference data
print(f'Uploading {len(inference_alerts)} unlabelled inference alerts...')
success2, errors2 = bulk(es, generate_actions(inference_alerts, INFERENCE_INDEX),
                         chunk_size=100, raise_on_error=False)
print(f'   Inference index: {success2} documents indexed, {len(errors2)} errors')

es.indices.refresh(index=TRAINING_INDEX)
es.indices.refresh(index=INFERENCE_INDEX)
print()
print('Both indices ready in Elastic Cloud.')


## Step 8 — Kibana Step-by-Step: Create the Classification Job

Now we use Elastic's **Data Frame Analytics** to train the classification model.
Follow these steps exactly in Kibana.

---

### PART A — Create Data Views for Both Indices

**Training data view:**
```
Stack Management > Data Views > Create data view
Name          : Alert Training Data
Index pattern : aiops-alert-training
Timestamp     : @timestamp
Save
```

**Inference data view (for results later):**
```
Stack Management > Data Views > Create data view
Name          : Alert Inference Data
Index pattern : aiops-alert-new
Timestamp     : @timestamp
Save
```

---

### PART B — Explore the Training Data in Discover

Before creating the ML job, spend 5 minutes understanding the training data.

1. Kibana > **Discover** > select `Alert Training Data`
2. Add columns: `incident_priority`, `cpu_pct`, `memory_pct`,
   `error_rate`, `affected_services`
3. Sort by `incident_priority` — you can see the P1, P2, P3 labels clearly
4. KQL filter: `incident_priority : "P1"` — notice how P1 alerts look different

> **Teaching question:** Can you see the pattern just by reading the data?
> A human can do this for 10-20 alerts. The ML does it for 500 simultaneously
> and captures patterns too subtle for human reading.

---

### PART C — Create the Data Frame Analytics Classification Job

**Step C1 — Open Data Frame Analytics:**
```
Machine Learning > Data Frame Analytics > Create job
```

**Step C2 — Select source:**
```
Source index   : aiops-alert-training
Click: Next
```

**Step C3 — Select job type:**
```
Job type: Classification
(Classification predicts a category — P1, P2, or P3 in our case)
Click: Next
```

**Step C4 — Configure the analysis:**

| Setting | Value | Explanation |
|---------|-------|-------------|
| Dependent variable | `incident_priority` | This is the LABEL field — what we want to predict |
| Training percent | `80` | 80% of data trains the model, 20% tests it |
| Prediction field name | `predicted_priority` | Where the model writes its prediction |
| Top classes | `3` | Show probability for P1, P2, and P3 |

**Step C5 — Select included fields:**

Include these fields (deselect everything else):
```
cpu_pct                  ✅
memory_pct               ✅
disk_pct                 ✅
error_rate               ✅
affected_services        ✅
response_time_ms         ✅
hour_of_day              ✅
is_business_hours        ✅
server_type              ✅
alert_source             ✅
previous_incidents_7d    ✅
incident_priority        ✅  (this is the label - must be included)

EXCLUDE these: alert_id, host_name, @timestamp, actual_downtime_min,
               is_training_data, day_of_week
```

**Step C6 — Job details:**
```
Job ID          : alert-priority-classifier
Description     : Classifies incoming IT alerts as P1, P2, or P3
Destination index: aiops-alert-results
```

**Step C7 — Create and start:**
```
Click: Create → Start
The job will take 60-120 seconds to train
Watch the progress bar complete
```

---

### PART D — Evaluate the Model

After the job completes, click **View model evaluation**.

**Confusion Matrix — What to Look For:**

The confusion matrix shows how accurately the model classified each priority
on the 20% test data it had never seen during training.

```
                  Predicted P1   Predicted P2   Predicted P3
Actual P1    |      High %    |    Low %     |    Low %     |
Actual P2    |      Low %     |    High %    |    Low %     |
Actual P3    |      Low %     |    Low %     |    High %    |
```

The diagonal (top-left to bottom-right) should be dark/high — these are
correct predictions. Off-diagonal cells are misclassifications.

**Key metric to note:** The **recall for P1** (what % of actual P1 incidents
were correctly identified as P1). This is the most important metric in
operations — a missed P1 is far more costly than a false P1 alarm.

**Expected results with this dataset:**
- Overall accuracy: ~85-92%
- P1 recall: ~80-90%
- P3 precision: ~88-95%

---

### PART E — View Feature Importance

After evaluation, click **Feature importance** tab.

This shows which features the model found most useful for making predictions.

> **Expected finding:** `error_rate`, `affected_services`, and `cpu_pct`
> will typically rank as the top 3 most important features.

> **Discussion question:** Does this match your intuition as an IT engineer?
> If `hour_of_day` appears in the top 5, what does that tell you?

---
##Testing the Trained Model:**
```
Search for Trained Model
Click: 'Test model' tab
Paste the JSON format alert from Test Case Cells in RAW document section
Click: Simulate Pipeline
Check for 'preicted priority' in the result section
```


# 1.Test Case for a P1 Prediction (Critical Outage)
This scenario represents a high-criticality database crash during core business hours with massive cascading dependencies and high error rates.

In [ ]:

[
  {
    "_source": {
      "alert_id": "TEST-P1-001",
      "@timestamp": "2026-06-23T10:15:00.000000+00:00",
      "host_name": "SRV-PROD-DB01",
      "server_type": "database_server",
      "alert_source": "system_health",
      "cpu_pct": 99.4,
      "memory_pct": 97.8,
      "disk_pct": 89.2,
      "error_rate": 95.1,
      "affected_services": 14,
      "response_time_ms": 8200.5,
      "hour_of_day": 10,
      "is_business_hours": true,
      "day_of_week": "Tuesday",
      "previous_incidents_7d": 4,
      "is_training_data": false
    }
  }
]


# 2. Test Case for a P2 Prediction (High/Degraded Performance)
This scenario represents a typical application memory leak or processing backlog. It's causing sluggishness and throwing errors outside of main business hours, impacting a couple of connected services.

In [ ]:
[
  {
    "_source": {
      "alert_id": "TEST-P2-001",
      "@timestamp": "2026-06-23T21:30:00.000000+00:00",
      "host_name": "SRV-PROD-APP04",
      "server_type": "app_server",
      "alert_source": "memory_monitor",
      "cpu_pct": 74.2,
      "memory_pct": 95.1,
      "disk_pct": 42.0,
      "error_rate": 31.4,
      "affected_services": 3,
      "response_time_ms": 2100.2,
      "hour_of_day": 21,
      "is_business_hours": false,
      "day_of_week": "Tuesday",
      "previous_incidents_7d": 1,
      "is_training_data": false
    }
  }
]

# 3. Test Case for a P3 Prediction (Low/Minor Warning)
This scenario represents a minor disk space warning on a staging web node. It features very low CPU metrics, zero affected services, and normal operational response timings.

In [ ]:
[
  {
    "_source": {
      "alert_id": "TEST-P3-001",
      "@timestamp": "2026-06-23T14:05:00.000000+00:00",
      "host_name": "SRV-STAGE-WEB02",
      "server_type": "web_server",
      "alert_source": "disk_monitor",
      "cpu_pct": 12.5,
      "memory_pct": 34.1,
      "disk_pct": 86.5,
      "error_rate": 0.0,
      "affected_services": 0,
      "response_time_ms": 45.8,
      "hour_of_day": 14,
      "is_business_hours": true,
      "day_of_week": "Tuesday",
      "previous_incidents_7d": 0,
      "is_training_data": false
    }
  }
]

## Step 9 — Run All 50 New Alerts at Once

This cell uses the Elasticsearch inference API to classify all 50 new alerts
in a single call. This is how a real production pipeline would work —
new alerts flow in, the model predicts their priority, and the enriched
alert is written to Elasticsearch for the operations team.

> **Note:** Run this cell AFTER the Data Frame Analytics job has completed
> and the model has been deployed in Kibana.


In [ ]:
# NOTE: Run this AFTER completing the Kibana steps above
# The model must be trained and deployed before inference will work
from elasticsearch.helpers import bulk

actions = []

for alert in inference_alerts:
    # Build a standard document containing your alert metadata
    doc = {
        "_index": "microland-classified-alerts",
        "_source": {
            "alert_id": alert['alert_id'],
            "host.name": alert['host_name'],
            "server_type": alert['server_type'],
            "cpu_pct": alert['cpu_pct'],
            "memory_pct": alert['memory_pct'],
            "disk_pct": alert['disk_pct'],
            "error_rate": alert['error_rate'],
            "affected_services": alert['affected_services'],
            "response_time_ms": alert['response_time_ms'],
            "hour_of_day": alert['hour_of_day'],
            "is_business_hours": alert['is_business_hours'],
            "previous_incidents_7d": alert['previous_incidents_7d']
        }
    }
    actions.append(doc)

print(f"Sending 50 alerts through Ingest Pipeline [incident-classification-pipeline]...")

# Execute the bulk upload while passing the PIPELINE name
success, errors = bulk(
    es,
    actions,
    pipeline="Your Ingest Pipeline name"  # <-- Use your pipeline name here!
)

es.indices.refresh(index="microland-classified-alerts")
print(f"Successfully processed and indexed {success} documents with AI classification fields!")

## Step 10 — View the Prediction in Kibana

### Create Data View

```
Stack Management > Data Views > Create data view
Name          : Classification Prediction Result
Index pattern : microland-classified-alerts
Timestamp     : @timestamp
Save
```




# Step 11: Print the Classification Prediction

In [ ]:
import pandas as pd

# Fetch the records from your results index
response = es.search(
    index="microland-classified-alerts",
    size=50,
    query={"match_all": {}}
)

verified_results = []
for hit in response['hits']['hits']:
    source = hit['_source']

    # Navigate down the exact nested object path: ml -> inference -> predicted_priority
    ml_layer = source.get('ml', {})
    inference_layer = ml_layer.get('inference', {})
    priority_layer = inference_layer.get('predicted_priority', {})

    # Extract the final string value and the confidence score
    predicted_priority = priority_layer.get('predicted_priority', 'N/A')
    confidence = priority_layer.get('prediction_score') or priority_layer.get('prediction_probability') or 0.0

    verified_results.append({
        "Alert ID": source.get("alert_id"),
        "CPU %": source.get("cpu_pct"),
        "Error Rate": source.get("error_rate"),
        "Affected Services": source.get("affected_services"),
        "PREDICTED PRIORITY": predicted_priority,
        "Confidence %": round(confidence * 100, 1) if confidence <= 1.0 else round(confidence, 1)
    })

# Convert to DataFrame for a clean tabular display in Colab
df_verification = pd.DataFrame(verified_results)

print("=== VERIFYING PIPELINE INFERENCE RESULTS ===")
print(df_verification.to_string(index=False))

## Step 12 — Visualise Prediction Results

If inference ran successfully, this cell visualises the model's predictions
alongside the key features — showing why the model made each decision.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Fetch the records from your target results index
response = es.search(
    index="microland-classified-alerts",
    size=50,
    query={"match_all": {}}
)

verified_results = []
for hit in response['hits']['hits']:
    source = hit['_source']

    # Drill down into the exact Elastic nested namespace path
    ml_layer = source.get('ml', {})
    inference_layer = ml_layer.get('inference', {})
    priority_layer = inference_layer.get('predicted_priority', {})

    # Extract structural inference outputs
    predicted_priority = priority_layer.get('predicted_priority', 'N/A')
    confidence = priority_layer.get('prediction_score') or priority_layer.get('prediction_probability') or 0.0

    # Append flattened schema for Pandas parsing
    verified_results.append({
        'alert_id': source.get('alert_id'),
        'server': source.get('host.name') or source.get('host_name') or 'Unknown',
        'cpu_pct': source.get('cpu_pct', 0.0),
        'error_rate': source.get('error_rate', 0.0),
        'affected_services': source.get('affected_services', 0),
        'predicted_priority': predicted_priority,
        'confidence_pct': round(confidence * 100, 1) if confidence <= 1.0 else round(confidence, 1)
    })

if verified_results:
    df_r = pd.DataFrame(verified_results)

    # Dynamically calculate metric counts for the summary bar charts
    pred_p1 = (df_r['predicted_priority'] == 'P1').sum()
    pred_p2 = (df_r['predicted_priority'] == 'P2').sum()
    pred_p3 = (df_r['predicted_priority'] == 'P3').sum()

    colors_pred = {'P1': '#C55A11', 'P2': '#0D6E6E', 'P3': '#1B3A6B'}

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # 1. Prediction distribution chart
    priorities = ['P1', 'P2', 'P3']
    counts = [pred_p1, pred_p2, pred_p3]
    bar_colors = [colors_pred[p] for p in priorities]
    axes[0].bar(priorities, counts, color=bar_colors, alpha=0.85, width=0.5)
    axes[0].set_title('Predicted Priority Distribution\n(50 new alerts)',
                      fontsize=11, fontweight='bold')
    axes[0].set_ylabel('Number of Alerts')
    for i, (p, c) in enumerate(zip(priorities, counts)):
        axes[0].text(i, c + 0.3, str(c), ha='center', fontweight='bold', fontsize=12)
    axes[0].grid(True, alpha=0.3, axis='y')

    # 2. CPU vs Error Rate scatter plot colored by pipeline classification
    for priority in ['P3', 'P2', 'P1']:
        subset = df_r[df_r['predicted_priority'] == priority]
        axes[1].scatter(subset['cpu_pct'], subset['error_rate'],
                        c=colors_pred[priority], label=priority, alpha=0.8, s=60)
    axes[1].set_xlabel('CPU %', fontsize=10)
    axes[1].set_ylabel('Error Rate (errors/min)', fontsize=10)
    axes[1].set_title('CPU % vs Error Rate\nColoured by Predicted Priority',
                      fontsize=11, fontweight='bold')
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3)

    # 3. Confidence score distribution histogram
    for priority in ['P1', 'P2', 'P3']:
        subset = df_r[df_r['predicted_priority'] == priority]['confidence_pct']
        if len(subset) > 0:
            axes[2].hist(subset, bins=10, alpha=0.65,
                         color=colors_pred[priority], label=priority)
    axes[2].set_xlabel('Model Confidence %', fontsize=10)
    axes[2].set_ylabel('Alert Count', fontsize=10)
    axes[2].set_title('Prediction Confidence by Priority\nHigher = more certain',
                      fontsize=11, fontweight='bold')
    axes[2].legend(fontsize=9)
    axes[2].grid(True, alpha=0.3)

    plt.suptitle('Alert Priority Classification Results — Elastic Data Frame Analytics Pipeline',
                 fontsize=12, fontweight='bold', color='#1B3A6B')
    plt.tight_layout()
    plt.show()

    # Print out isolated P1 alerts (the most critical infrastructure conditions)
    p1_alerts = df_r[df_r['predicted_priority'] == 'P1'].sort_values(
        'confidence_pct', ascending=False)
    if len(p1_alerts) > 0:
        print(f'P1 Critical alerts predicted ({len(p1_alerts)} total):')
        print(p1_alerts[['alert_id', 'server', 'cpu_pct', 'error_rate',
                          'affected_services', 'confidence_pct']].to_string(index=False))
else:
    print('No data returned from target index. Ensure pipeline data ingestion step completed.')